# 03 — Entanglement Analysis: Von Neumann Entropy & Mutual Information

> **spinq-vqe** | ARPA Quantum Logical Systems (QONDRA)

## Objective

Analyse the entanglement structure of the VQE ground-state wavefunctions from Notebook 02.

1. **Von Neumann entropy** $S(\rho_A) = -\mathrm{Tr}(\rho_A \log_2 \rho_A)$ for all bipartitions.
2. **Mutual information** $I(i:j) = S(i) + S(j) - S(ij)$ between all site pairs.

### References
- Vidal et al. (2003) PRL 90  
- Depenbrock et al. (2012) PRL 109 — Kagome entanglement spectrum  
- Eisert et al. (2010) Rev. Mod. Phys. — area laws

In [ ]:
from __future__ import annotations

import os
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import matplotlib.pyplot as plt
import pennylane as qp

from spinq_vqe import kagome, entanglement

os.makedirs('../figures', exist_ok=True)

PAL = {'hea': '#7EB8D4', 'mera': '#E8A598', 'A': '#B8B8E8', 'B': '#F5C9A0', 'C': '#A8D8B0'}

plt.rcParams.update({
    'font.family': 'sans-serif', 'font.size': 11,
    'figure.facecolor': 'white', 'axes.facecolor': '#FAFAFA',
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.color': '#EBEBEB',
    'grid.linewidth': 0.7, 'figure.dpi': 110,
})

print('PennyLane', qp.__version__)

---
## 1. Load Statevectors

In [ ]:
sv_hea  = np.load('../data/statevector_hea_best.npy')
sv_mera = np.load('../data/statevector_mera_best.npy')

G9      = kagome.kagome_graph(n_cells=3)
N_SITES = kagome.n_sites(G9)
partition = kagome.sublattice_partition(G9)

print(f'HEA  sv: shape={sv_hea.shape}  norm={np.linalg.norm(sv_hea):.6f}')
print(f'MERA sv: shape={sv_mera.shape}  norm={np.linalg.norm(sv_mera):.6f}')
print(f'Sublattice A: {partition[0]}')
print(f'Sublattice B: {partition[1]}')
print(f'Sublattice C: {partition[2]}')

---
## 2. Von Neumann Entropy — Bipartition Scan

In [ ]:
cut_sizes = list(range(1, N_SITES))
svn_hea, svn_mera = [], []

for k in cut_sizes:
    region_A = list(range(k))
    rho_h = entanglement.reduced_density_matrix(sv_hea,  region_A, N_SITES)
    rho_m = entanglement.reduced_density_matrix(sv_mera, region_A, N_SITES)
    svn_hea.append(entanglement.von_neumann_entropy(rho_h))
    svn_mera.append(entanglement.von_neumann_entropy(rho_m))

print('Von Neumann entropy S(A) — bipartition scan')
print(f'{"Cut |A|":>8}  {"HEA":>10}  {"MERA":>10}')
print('─' * 32)
for k, sh, sm in zip(cut_sizes, svn_hea, svn_mera):
    print(f'{k:>8}  {sh:>10.6f}  {sm:>10.6f}')

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(cut_sizes, svn_hea,  'o-', color=PAL['hea'],  linewidth=2, markersize=6, label='HEA')
ax.plot(cut_sizes, svn_mera, 's-', color=PAL['mera'], linewidth=2, markersize=6, label='MERA')
for x in [3, 6]:
    ax.axvline(x, color='#CCCCCC', linestyle=':', linewidth=1.2)
ylo = ax.get_ylim()[0]
for cx, lbl in [(1.5, 'cell 0'), (4.5, 'cell 1'), (7.5, 'cell 2')]:
    ax.text(cx, ylo, lbl, ha='center', fontsize=8, color='#999999')
ax.set_xlabel('Cut position |A|', color='#555555')
ax.set_ylabel('S(A) [bits]', color='#555555')
ax.set_title('Entanglement Entropy Profile — 9-site Kagome',
             fontsize=11, fontweight='semibold', color='#333333')
ax.set_xticks(cut_sizes)
ax.legend(framealpha=0.9, edgecolor='#DDDDDD')
plt.tight_layout()
plt.savefig('../figures/entanglement_profile.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → figures/entanglement_profile.png')

---
## 3. Pairwise Mutual Information Matrix

In [ ]:
print('Computing 9×9 mutual information matrices ...')
MI_hea  = entanglement.mutual_information_matrix(sv_hea,  N_SITES)
MI_mera = entanglement.mutual_information_matrix(sv_mera, N_SITES)
print('Done.')

intra = [(0,1),(0,2),(1,2),(3,4),(3,5),(4,5),(6,7),(6,8),(7,8)]
inter = [(2,3),(5,6)]

for name, MI in [('HEA', MI_hea), ('MERA', MI_mera)]:
    i_intra = np.mean([MI[a,b] for a,b in intra])
    i_inter = np.mean([MI[a,b] for a,b in inter])
    ratio = i_intra / i_inter if i_inter > 1e-9 else float('inf')
    print(f'{name}  intra-triangle MI={i_intra:.4f}  inter-cell MI={i_inter:.4f}  ratio={ratio:.2f}×')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
sl_map = {0: 'A', 1: 'B', 2: 'C'}

for ax, MI, title in [(axes[0], MI_hea, 'HEA'), (axes[1], MI_mera, 'MERA')]:
    im = ax.imshow(MI, cmap='YlOrBr', aspect='equal', vmin=0)
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    for i in range(N_SITES):
        ax.text(i, i, sl_map[G9.nodes[i]['sublattice']],
                ha='center', va='center', fontsize=9, color='#444444', fontweight='semibold')
    ax.set_xticks(range(N_SITES))
    ax.set_yticks(range(N_SITES))
    ax.set_xticklabels(list(range(N_SITES)), fontsize=8)
    ax.set_yticklabels(list(range(N_SITES)), fontsize=8)
    for k in range(3):
        rect = plt.Rectangle((3*k - 0.5, 3*k - 0.5), 3, 3,
                              linewidth=1.5, edgecolor='#AAAAAA', facecolor='none', linestyle='--')
        ax.add_patch(rect)
    ax.set_title(f'Mutual Information I(i:j) — {title}',
                 fontsize=11, fontweight='semibold', color='#333333')

plt.suptitle('Quantum Correlations: Pairwise Mutual Information',
             fontsize=13, fontweight='semibold', color='#333333', y=1.01)
plt.tight_layout()
plt.savefig('../figures/mutual_information.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → figures/mutual_information.png')

---
## 4. Single-Site Entropies

In [ ]:
sl_labels = {0: 'A', 1: 'B', 2: 'C'}
sl_colors = {0: PAL['A'], 1: PAL['B'], 2: PAL['C']}

ss_hea, ss_mera = [], []
print(f'{"Site":>5}  {"Sublt":>5}  {"HEA S":>8}  {"MERA S":>8}')
print('─' * 34)
for i in range(N_SITES):
    rho_h = entanglement.reduced_density_matrix(sv_hea,  [i], N_SITES)
    rho_m = entanglement.reduced_density_matrix(sv_mera, [i], N_SITES)
    sh = entanglement.von_neumann_entropy(rho_h)
    sm = entanglement.von_neumann_entropy(rho_m)
    ss_hea.append(sh)
    ss_mera.append(sm)
    print(f'{i:>5}  {sl_labels[G9.nodes[i]["sublattice"]]:>5}  {sh:>8.4f}  {sm:>8.4f}')

fig, ax = plt.subplots(figsize=(8, 3.5))
x = np.arange(N_SITES)
w = 0.35
ax.bar(x - w/2, ss_hea,  w, color=PAL['hea'],  edgecolor='#AAAAAA', lw=0.5, label='HEA')
ax.bar(x + w/2, ss_mera, w, color=PAL['mera'], edgecolor='#AAAAAA', lw=0.5, label='MERA')
ax.set_xlabel('Site index', color='#555555')
ax.set_ylabel('S({i}) [bits]', color='#555555')
ax.set_title('Single-Site Entanglement Entropies — 9-site Kagome',
             fontsize=11, fontweight='semibold', color='#333333')
ax.set_xticks(x)
ax.set_xticklabels([f'{i}\n({sl_labels[G9.nodes[i]["sublattice"]]})' for i in range(N_SITES)])
ax.legend(framealpha=0.9, edgecolor='#DDDDDD')
for bnd in [2.5, 5.5]:
    ax.axvline(bnd, color='#CCCCCC', linestyle=':', linewidth=1)
plt.tight_layout()
plt.savefig('../figures/single_site_entropy.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → figures/single_site_entropy.png')

---
## 5. Observations

| Observable | Expected | Interpretation |
|-----------|----------|----------------|
| S(A) profile | Peaks at mid-chain | Area law with boundary contribution |
| Intra-triangle MI >> inter-cell | Ratio > 2x | Frustrated triangles entangle strongly |
| Single-site S ~1 bit | Max for qubit | Near-maximal local entanglement |
| MERA > HEA intra-triangle | Geometry-matched ansatz | MERA respects Kagome structure |

---
*spinq-vqe / ARPA QONDRA — Notebook 03*